# YOLO ONNX To Horizon Bin Workflow

这个 notebook 用于把 YOLO 导出的 `best.onnx` 转成 Horizon X5 的 `bin`。

默认目录按你们教学流程设计：
- `OE_ROOT / yolo11/` 下放 `best.onnx`
- `OE_ROOT / yolo11/` 下放 `generate_calibration_data.py`
- `OE_ROOT / yolo11/` 下放 `yolo11_detect_bayese_640x640_nv12.yaml`
- `OE_ROOT / yolo11/JPEGImages/` 下放校准图片


In [ ]:
from pathlib import Path
import subprocess
import shlex

OE_ROOT = Path('/data/horizon/horizon_x5_open_explorer_v1.2.8-py310_20240926')
WORKDIR = OE_ROOT / 'yolo11'
ONNX_FILE = 'best.onnx'
CONFIG_FILE = 'yolo11_detect_bayese_640x640_nv12.yaml'
GEN_CAL_SCRIPT = 'generate_calibration_data.py'
JPEG_DIR = WORKDIR / 'JPEGImages'
DOCKER_IMAGE = 'openexplorer/ai_toolchain_ubuntu_20_x5_cpu:v1.2.8-py310'
DATASET_MOUNT = '/data'
MARCH = 'bayes-e'

print('WORKDIR =', WORKDIR)


def run(cmd: str):
    print(f'\n$ {cmd}\n')
    completed = subprocess.run(cmd, shell=True, executable='/bin/bash', text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if completed.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {completed.returncode}')
    return completed


## 1. 检查输入


In [ ]:
assert OE_ROOT.exists(), f'OE_ROOT not found: {OE_ROOT}'
assert WORKDIR.exists(), f'WORKDIR not found: {WORKDIR}'
assert (WORKDIR / ONNX_FILE).exists(), f'ONNX not found: {WORKDIR / ONNX_FILE}'
assert (WORKDIR / CONFIG_FILE).exists(), f'Config not found: {WORKDIR / CONFIG_FILE}'
assert (WORKDIR / GEN_CAL_SCRIPT).exists(), f'Calibration script not found: {WORKDIR / GEN_CAL_SCRIPT}'
assert JPEG_DIR.exists(), f'JPEGImages not found: {JPEG_DIR}'
print('JPEG count =', len(list(JPEG_DIR.glob('*.jpg'))))


## 2. 生成校准数据


In [ ]:
run(f'cd {shlex.quote(str(WORKDIR))} && python3 {GEN_CAL_SCRIPT} --src JPEGImages/')


## 3. checker


In [ ]:
cmd = f"sudo docker run --rm -v {OE_ROOT}:/open_explorer -v {DATASET_MOUNT}:/data/horizon_x5/data {DOCKER_IMAGE} bash -lc 'cd /open_explorer/yolo11 && hb_mapper checker --model-type onnx --march {MARCH} --model ./{ONNX_FILE}'"
run(cmd)


## 4. 生成 bin


In [ ]:
cmd = f"sudo docker run --rm -v {OE_ROOT}:/open_explorer -v {DATASET_MOUNT}:/data/horizon_x5/data {DOCKER_IMAGE} bash -lc 'cd /open_explorer/yolo11 && hb_mapper makertbin --model-type onnx --config {CONFIG_FILE}'"
run(cmd)


## 5. 查看 bin 产物


In [ ]:
output_dir = WORKDIR / 'model_output'
assert output_dir.exists(), f'model_output not found: {output_dir}'
for p in sorted(output_dir.iterdir()):
    print(' -', p.name, f'({p.stat().st_size / 1024 / 1024:.2f} MB)')


## 6. 可选：性能检查和模型优化


In [ ]:
# 先在上一格确认生成的 bin 文件名，再修改下面两个变量
BIN_NAME = 'yolo11n_detect_bayese_640x640_nv12.bin'
MODIFIED_BIN = 'yolo11n_detect_bayese_640x640_nv12_modified.bin'

print('示例命令：')
print(f"hb_perf {BIN_NAME}")
print(
    'hb_model_modifier ' + BIN_NAME +
    ' -r /model.23/cv2.0/cv2.0.2/Conv_output_0_HzDequantize' +
    ' -r /model.23/cv2.1/cv2.1.2/Conv_output_0_HzDequantize' +
    ' -r /model.23/cv2.2/cv2.2.2/Conv_output_0_HzDequantize'
)
print(f"hb_perf {MODIFIED_BIN}")
